Qwen3-VL 4B Instruct + Unsloth + optional MediaPipe pose

**Notebook vs local scripts:** You do **not** have to use a notebook. The same flow runs as plain Python:

- `train_qwen3_vl_4b.py` / `infer_qwen3_vl_4b.py` with `--pose_mode` etc.

Use this notebook when you want **interactive** runs (Colab, Jupyter, or VS Code). Use the scripts for **automation**, CI, or headless servers.

**GPU:** Training expects **CUDA** (Unsloth + 4-bit). Colab T4 works; local Linux + NVIDIA works too.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!unzip /content/drive/MyDrive/backend/backend_vlm_colab.zip -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0018_003_5086.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0013_001_1562.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0028_003_3141.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0018_001_469.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0018_003_5092.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0019_002_12178.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0013_001_1576.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0013_002_2979.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0011_002_17130.jpg  
  inflating: /content/backend/data/FineBadminton-master/dataset/image/0011_004_22205.jpg  
  inflating: /content/backend/data/FineBadminto

### Colab: upload `backend` from your Mac

From your machine (repo root), zip **only** `backend` (excludes heavy `venv`):

```bash
cd /path/to/IsoCourt && zip -r backend.zip backend -x "backend/venv/*" -x "backend/**/__pycache__/*" -x "backend/**/*.pyc"
```

In Colab: **Upload** `backend.zip`, then in a cell:

```python
!unzip -q -o backend.zip -d /content
```

You should get `/content/backend/pipelines/vlm/qwen3_vl_config.py`.

### Paths and imports

The next cell finds `backend/pipelines/vlm` whether you uploaded **only `backend`** (`/content/backend/...`) or the **full repo** (`/content/IsoCourt/backend/...`). You can override with `VLM_DIR = Path("...")` or env `ISOCOURT_VLM_DIR`.

In [5]:
import os
import sys
from pathlib import Path
from typing import Optional

_VLM_MARKER = "qwen3_vl_config.py"


def _is_vlm_dir(p: Path) -> bool:
    return p.is_dir() and (p / _VLM_MARKER).is_file()


def _vlm_under_backend_root(backend_root: Path) -> Optional[Path]:
    cand = (backend_root / "pipelines" / "vlm").resolve()
    return cand if _is_vlm_dir(cand) else None


def _vlm_under_repo_root(repo_root: Path) -> Optional[Path]:
    cand = (repo_root / "backend" / "pipelines" / "vlm").resolve()
    return cand if _is_vlm_dir(cand) else None


def _search_bases(cwd: Path):
    seen: set[Path] = set()
    for b in _iter_search_roots(cwd):
        b = b.resolve()
        if b not in seen:
            seen.add(b)
            yield b


def _iter_search_roots(cwd: Path):
    yield cwd
    yield cwd / "IsoCourt"
    content = Path("/content")
    if content.is_dir():
        yield content / "backend"
        try:
            for child in content.iterdir():
                if child.is_dir():
                    yield child
        except OSError:
            pass
    try:
        for child in cwd.iterdir():
            if child.is_dir():
                yield child
    except OSError:
        pass
    drive = cwd / "drive" / "MyDrive"
    if drive.is_dir():
        try:
            for a in drive.iterdir():
                if not a.is_dir():
                    continue
                yield a
                try:
                    for b in a.iterdir():
                        if b.is_dir():
                            yield b
                except OSError:
                    pass
        except OSError:
            pass


def _resolve_vlm_dir() -> Path:
    env = os.environ.get("ISOCOURT_VLM_DIR") or os.environ.get("VLM_DIR")
    if env:
        p = Path(env).expanduser().resolve()
        if _is_vlm_dir(p):
            return p

    cwd = Path.cwd().resolve()
    if _is_vlm_dir(cwd):
        return cwd

    for base in _search_bases(cwd):
        hit = _vlm_under_backend_root(base)
        if hit is not None:
            return hit
        hit = _vlm_under_repo_root(base)
        if hit is not None:
            return hit

    for parent in cwd.parents:
        hit = _vlm_under_backend_root(parent)
        if hit is not None:
            return hit
        hit = _vlm_under_repo_root(parent)
        if hit is not None:
            return hit

    return cwd


# Optional: force path if auto-detect fails (Colab after unzip to /content):
# VLM_DIR = Path("/content/backend/pipelines/vlm")

VLM_DIR = _resolve_vlm_dir()

BACKEND_ROOT = VLM_DIR.parent.parent
for p in (VLM_DIR, BACKEND_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

if not _is_vlm_dir(VLM_DIR):
    raise FileNotFoundError(
        f"Could not find {_VLM_MARKER}. Tried backend-only layout (/content/backend/pipelines/vlm) "
        f"and full repo (/content/…/backend/pipelines/vlm). cwd={Path.cwd()!s}. "
        f"Set VLM_DIR = Path(\"/content/backend/pipelines/vlm\") or env ISOCOURT_VLM_DIR."
    )

print("VLM_DIR:", VLM_DIR)
print("BACKEND_ROOT:", BACKEND_ROOT)

VLM_DIR: /content/backend/pipelines/vlm
BACKEND_ROOT: /content/backend


### Install dependencies

**Local:** `pip install -r requirements-unsloth-vlm.txt` in your venv (from `backend/pipelines/vlm`).

**Colab:** uncomment and run the pip cell below once per runtime.

In [6]:
# Run once per fresh runtime.
# If this notebook kernel already has deps installed, you can skip.
%pip install -U pip
%pip install unsloth
%pip install "transformers>=4.57.0" "trl>=0.22.0" datasets accelerate bitsandbytes peft sentencepiece protobuf pillow mediapipe opencv-python

# Optional HF login if model pulls fail with auth errors:
# from huggingface_hub import login
# login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 50.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 114.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 66.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 69.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 141.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninsta

### Configuration

**Pose landmarker:** Download `pose_landmarker_lite.task` from [MediaPipe pose landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker) and set `POSE_MODEL_PATH` to the file, or place it at `backend/models/pose_landmarker_lite.task` (same as `PoseEstimator` default).

In [15]:
from qwen3_vl_config import DEFAULT_MAX_SEQ_LENGTH, DEFAULT_MODEL_ID

MODEL_NAME = DEFAULT_MODEL_ID  # unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit
OUTPUT_DIR = VLM_DIR / "outputs" / "notebook_qwen3vl4b"
# Regenerate: python build_finebadminton_jsonl.py (from this folder)
JSONL_PATH = BACKEND_ROOT / "data" / "FineBadminton-master" / "dataset" / "finebadminton_vlm_train.jsonl"

# MediaPipe: none | overlay | text | both
POSE_MODE = "both"
POSE_MODEL_PATH = BACKEND_ROOT / "models" / "pose_landmarker_lite.task"
# Upscale before MediaPipe if min(h,w) < this (720p wide shots → ~1.33×). Set 0 to disable.
POSE_MIN_SHORT_EDGE = 960

MAX_SEQ_LENGTH = DEFAULT_MAX_SEQ_LENGTH
MAX_STEPS = None  # set e.g. 30 for a quick smoke test; None = use NUM_EPOCHS only
NUM_EPOCHS = 60

assert JSONL_PATH.exists(), f"JSONL not found: {JSONL_PATH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    POSE_MIN_SHORT_EDGE
except NameError:
    POSE_MIN_SHORT_EDGE = 960  # add above if you edit this cell: # POSE_MIN_SHORT_EDGE = 960

print("JSONL_PATH:", JSONL_PATH)
print("POSE_MODE:", POSE_MODE)
print("POSE_MODEL_PATH:", POSE_MODEL_PATH)
print("POSE_MIN_SHORT_EDGE:", POSE_MIN_SHORT_EDGE)

JSONL_PATH: /content/backend/data/FineBadminton-master/dataset/finebadminton_vlm_train.jsonl
POSE_MODE: both
POSE_MODEL_PATH: /content/backend/models/pose_landmarker_lite.task
POSE_MIN_SHORT_EDGE: 960


In [16]:
# Preflight checks: GPU + JSONL format + optional pose task file
import json
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No CUDA GPU. On Colab, switch Runtime -> GPU.")

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    first_line = f.readline().strip()
row = json.loads(first_line)
for key in ("image", "instruction", "response"):
    assert key in row, f"Missing key '{key}' in JSONL row"
print("JSONL keys OK")

if POSE_MODE != "none":
    if not POSE_MODEL_PATH.exists():
        print("Pose model missing. Downloading pose_landmarker_lite.task...")
        import urllib.request
        url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
        POSE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, POSE_MODEL_PATH)
    print("Pose model path:", POSE_MODEL_PATH)

CUDA available: True
GPU: Tesla T4
JSONL keys OK
Pose model path: /content/backend/models/pose_landmarker_lite.task


### Load model + LoRA

In [17]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required for Unsloth Qwen3-VL training in this notebook.")

from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

==((====))==  Unsloth 2026.3.17: Fast Qwen3_Vl patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

### Build training set (JSONL + optional MediaPipe pose)

Same behavior as `load_dataset_jsonl.load_jsonl_conversations(..., pose_mode=..., pose_model_path=...)`.

In [ ]:
import importlib

# Colab: editing .py on disk does NOT refresh imports — reload (or Runtime → Restart session).
import vlm_pose
importlib.reload(vlm_pose)
import load_dataset_jsonl
importlib.reload(load_dataset_jsonl)
from load_dataset_jsonl import load_jsonl_conversations_train_val, trainer_vision_kwargs

_pose_min = None if POSE_MIN_SHORT_EDGE == 0 else POSE_MIN_SHORT_EDGE
train_dataset, val_dataset = load_jsonl_conversations_train_val(
    str(JSONL_PATH),
    pose_mode=POSE_MODE,
    pose_model_path=str(POSE_MODEL_PATH) if POSE_MODEL_PATH else None,
    pose_min_short_edge=_pose_min,
)
len(train_dataset), len(val_dataset)

In [12]:
def _pose_label(sample: dict) -> str:
    for block in sample["messages"][0]["content"]:
        if block.get("type") == "text":
            t = block["text"]
            if "[Pose: no person detected.]" in t:
                return "miss"
            if "[Pose summary]" in t:
                return "hit"
            return "no_pose_prefix"
    return "no_text"

hits, misses, other = [], [], []
for i, s in enumerate(train_dataset):
    lab = _pose_label(s)
    if lab == "hit":
        hits.append(i)
    elif lab == "miss":
        misses.append(i)
    else:
        other.append((i, lab))

print(
    f"pose hits: {len(hits)} | no detection: {len(misses)} | other: {len(other)} | total: {len(train_dataset)}"
)

# First few lines that *did* get a pose
for idx in hits[:5]:
    for block in train_dataset[idx]["messages"][0]["content"]:
        if block.get("type") == "text":
            txt = block["text"]
            print(f"\n--- index {idx} (first 400 chars) ---\n{txt[:400]}...")
            break

# First few misses (optional)
for idx in misses[:3]:
    print(f"\n--- miss index {idx} ---")

pose hits: 117 | no detection: 297 | other: 0 | total: 414

--- index 84 (first 400 chars) ---
[Pose summary] nose=(0.855,0.540,vis=1.00) L_shoulder=(0.886,0.536,vis=1.00) R_shoulder=(0.874,0.582,vis=1.00) L_elbow=(0.896,0.536,vis=0.89) R_elbow=(0.882,0.608,vis=0.87) L_wrist=(0.868,0.552,vis=0.88) R_wrist=(0.860,0.622,vis=0.90) L_hip=(0.959,0.592,vis=0.99) R_hip=(0.951,0.622,vis=0.99) L_knee=(0.967,0.603,vis=0.19) R_knee=(0.961,0.630,vis=0.28) L_ankle=(0.949,0.618,vis=0.07) R_ankle=(0.953,0...

--- index 87 (first 400 chars) ---
[Pose summary] nose=(0.842,0.547,vis=1.00) L_shoulder=(0.867,0.557,vis=0.99) R_shoulder=(0.859,0.585,vis=0.99) L_elbow=(0.861,0.555,vis=0.83) R_elbow=(0.859,0.600,vis=0.63) L_wrist=(0.814,0.554,vis=0.83) R_wrist=(0.826,0.596,vis=0.79) L_hip=(0.935,0.622,vis=0.96) R_hip=(0.929,0.639,vis=0.96) L_knee=(0.875,0.636,vis=0.47) R_knee=(0.874,0.653,vis=0.16) L_ankle=(0.828,0.662,vis=0.38) R_ankle=(0.827,0...

--- index 88 (first 400 chars) ---
[Pose summary] nose=(0.65

### Train

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

from vlm_train_metrics import build_sft_eval_compute_metrics

FastVisionModel.for_training(model)

tkwargs = trainer_vision_kwargs(max_length=MAX_SEQ_LENGTH)
train_kwargs = dict(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir=str(OUTPUT_DIR),
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    **tkwargs,
)
if MAX_STEPS is not None:
    train_kwargs["max_steps"] = MAX_STEPS
else:
    train_kwargs["num_train_epochs"] = NUM_EPOCHS

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=build_sft_eval_compute_metrics(tokenizer),
    args=SFTConfig(**train_kwargs),
)

trainer.train()

### Save LoRA adapter

In [ ]:
adapter_dir = OUTPUT_DIR / "lora_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved:", adapter_dir)

### Inference (optional)

Same behavior as `infer_qwen3_vl_4b.py`: optional pose overlay/text before the VLM.

In [ ]:
from PIL import Image
from transformers import TextStreamer

from vlm_pose import apply_pose_to_pil, create_pose_estimator

_fb = BACKEND_ROOT / "data" / "FineBadminton-master" / "dataset" / "image" / "0011_001_16363.jpg"
IMAGE_PATH = _fb if _fb.is_file() else (VLM_DIR / "example_data" / "sample.jpg")
USER_PROMPT = "Describe this image."
INFER_POSE_MODE = "none"  # or overlay / text / both

image = Image.open(IMAGE_PATH).convert("RGB")
prompt = USER_PROMPT
if INFER_POSE_MODE != "none":
    pe = create_pose_estimator(str(POSE_MODEL_PATH) if POSE_MODEL_PATH else None)
    _pm = None if POSE_MIN_SHORT_EDGE == 0 else POSE_MIN_SHORT_EDGE
    image, prompt = apply_pose_to_pil(
        image, pe, mode=INFER_POSE_MODE, instruction=prompt, min_short_edge_for_pose=_pm
    )

FastVisionModel.for_inference(model)

messages = [
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
)
device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=1.5,
    min_p=0.1,
)